# Notebook 4: Forecasting Experiments

Covers **Section 4** of the assignment:
- Train LSTM, TCN, Transformer with iterative hyperparameter tuning
- Evaluate on the week December 16–22 for three geographic areas
- Produce all required plots (9 total) and tables (3 total)
- Document all timing statistics

In [ ]:
import sys, os, json
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import torch

from data_loader import load_processed
from features import prepare_area_data, make_dataloaders
from models import LSTMForecaster, TCNForecaster, TransformerForecaster
from train import run_experiment, set_seed, get_device
from evaluate import (
    run_full_evaluation, build_results_table,
    plot_predictions, plot_failure_case, compute_all_metrics
)
from tuning import ExperimentLogger

PROCESSED_DIR   = '../data/processed/'
FIGURES_DIR     = '../report/figures/'
EXPERIMENTS_DIR = '../experiments/'
os.makedirs(FIGURES_DIR,     exist_ok=True)
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

set_seed(42)
device = get_device()

In [ ]:
# Load processed matrix and top-3 areas identified in Notebook 02
matrix = load_processed(os.path.join(PROCESSED_DIR, 'traffic_matrix.parquet'))

with open(os.path.join(PROCESSED_DIR, 'top3_areas.json')) as f:
    top3 = json.load(f)['top3']

TARGET_AREAS = top3
PRIMARY_AREA = top3[0]
print('Target areas:', TARGET_AREAS)
print('Primary tuning area:', PRIMARY_AREA)

## 4.1 LSTM — Iterative Hyperparameter Tuning

We tune on the highest-traffic area only, then apply the best config to all three.
Each experiment is logged with its rationale and the reasoning for what to try next.

In [ ]:
logger = ExperimentLogger(os.path.join(EXPERIMENTS_DIR, 'experiment_log.jsonl'))

# ============================================================
# LSTM Experiment 1 — baseline
# Rationale: 1-day lookback is the standard starting point in
# network traffic forecasting literature (Huang et al. 2017).
# Hidden=64 and 2 layers match the most common cited config.
# ============================================================
data_lstm1 = prepare_area_data(matrix, PRIMARY_AREA, lookback=144)
lstm_exp1  = run_experiment(
    model_class  = LSTMForecaster,
    model_kwargs = {'input_size': 1, 'hidden_size': 64, 'num_layers': 2, 'dropout': 0.2},
    data         = data_lstm1,
    n_epochs=100, lr=1e-3, batch_size=64, patience=10,
    output_dir=EXPERIMENTS_DIR, model_name='LSTM_exp01',
)

m1 = compute_all_metrics(np.array(lstm_exp1['targets_raw']), np.array(lstm_exp1['preds_raw']))
logger.log(
    experiment_id = 'LSTM_exp01',
    model_name    = 'LSTM',
    config        = {'hidden_size':64,'num_layers':2,'dropout':0.2,'lookback':144,'lr':1e-3,'bs':64},
    metrics       = m1,
    rationale     = 'Baseline from literature. 1-day lookback to capture daily cycle.',
    next_steps    = 'Fill after seeing results: if val loss plateaus early, increase hidden_size or lookback.',
)

In [ ]:
# ============================================================
# LSTM Experiment 2
# Rationale: ACF from EDA showed significant correlation at
# 2-day lag (288 steps). Expanding lookback + hidden_size to
# give the model more capacity to exploit this structure.
# ============================================================
data_lstm2 = prepare_area_data(matrix, PRIMARY_AREA, lookback=288)
lstm_exp2  = run_experiment(
    model_class  = LSTMForecaster,
    model_kwargs = {'input_size': 1, 'hidden_size': 128, 'num_layers': 2, 'dropout': 0.2},
    data         = data_lstm2,
    n_epochs=100, lr=5e-4, batch_size=64, patience=12,
    output_dir=EXPERIMENTS_DIR, model_name='LSTM_exp02',
)

m2 = compute_all_metrics(np.array(lstm_exp2['targets_raw']), np.array(lstm_exp2['preds_raw']))
logger.log(
    experiment_id = 'LSTM_exp02',
    model_name    = 'LSTM',
    config        = {'hidden_size':128,'num_layers':2,'dropout':0.2,'lookback':288,'lr':5e-4,'bs':64},
    metrics       = m2,
    rationale     = 'ACF showed strong autocorrelation at 2-day lag. Expanding lookback from 144→288.',
    next_steps    = 'Fill after seeing results.',
)

In [ ]:
# ============================================================
# LSTM Experiment 3 — refine based on best of exp1/2
# Update this cell's config after seeing exp1+exp2 results.
# ============================================================
data_lstm3 = prepare_area_data(matrix, PRIMARY_AREA, lookback=288)
lstm_exp3  = run_experiment(
    model_class  = LSTMForecaster,
    model_kwargs = {'input_size': 1, 'hidden_size': 128, 'num_layers': 2, 'dropout': 0.3},
    data         = data_lstm3,
    n_epochs=150, lr=5e-4, batch_size=64, patience=15,
    output_dir=EXPERIMENTS_DIR, model_name='LSTM_exp03',
)

m3 = compute_all_metrics(np.array(lstm_exp3['targets_raw']), np.array(lstm_exp3['preds_raw']))
logger.log(
    experiment_id = 'LSTM_exp03',
    model_name    = 'LSTM',
    config        = {'hidden_size':128,'num_layers':2,'dropout':0.3,'lookback':288,'lr':5e-4,'bs':64},
    metrics       = m3,
    rationale     = 'Increasing dropout to 0.3 to reduce overfitting seen in exp02 val curve.',
    next_steps    = 'This is the final LSTM config if val loss improves over exp02.',
)

## 4.2 TCN — Iterative Tuning

In [ ]:
# ============================================================
# TCN Experiment 1 — Bai et al. (2018) baseline config
# Receptive field = 1 + (3-1)*2*(1+2+4+8) = 61 steps (~10h)
# ============================================================
data_tcn1 = prepare_area_data(matrix, PRIMARY_AREA, lookback=144)
tcn_exp1  = run_experiment(
    model_class  = TCNForecaster,
    model_kwargs = {'input_size':1,'num_channels':[32,32,64,64],'kernel_size':3,'dropout':0.2},
    data         = data_tcn1,
    n_epochs=100, lr=1e-3, batch_size=64, patience=10,
    output_dir=EXPERIMENTS_DIR, model_name='TCN_exp01',
)

mt1 = compute_all_metrics(np.array(tcn_exp1['targets_raw']), np.array(tcn_exp1['preds_raw']))
logger.log(
    experiment_id = 'TCN_exp01',
    model_name    = 'TCN',
    config        = {'channels':[32,32,64,64],'kernel':3,'dropout':0.2,'lookback':144,'lr':1e-3},
    metrics       = mt1,
    rationale     = 'Bai et al. baseline. Receptive field ~61 steps (10h) — shorter than daily cycle.',
    next_steps    = 'Expand receptive field via larger kernel or more blocks if performance is limited.',
)

In [ ]:
# ============================================================
# TCN Experiment 2 — wider kernel, more channels
# RF = 1 + (5-1)*2*(1+2+4+8) = 121 steps (~20h)
# ============================================================
data_tcn2 = prepare_area_data(matrix, PRIMARY_AREA, lookback=144)
tcn_exp2  = run_experiment(
    model_class  = TCNForecaster,
    model_kwargs = {'input_size':1,'num_channels':[64,64,64,64],'kernel_size':5,'dropout':0.2},
    data         = data_tcn2,
    n_epochs=100, lr=1e-3, batch_size=64, patience=10,
    output_dir=EXPERIMENTS_DIR, model_name='TCN_exp02',
)

mt2 = compute_all_metrics(np.array(tcn_exp2['targets_raw']), np.array(tcn_exp2['preds_raw']))
logger.log(
    experiment_id = 'TCN_exp02',
    model_name    = 'TCN',
    config        = {'channels':[64,64,64,64],'kernel':5,'dropout':0.2,'lookback':144,'lr':1e-3},
    metrics       = mt2,
    rationale     = 'Wider kernel (3→5) and uniform 64 channels doubles RF to ~20h.',
    next_steps    = 'Fill after results.',
)

In [ ]:
# TCN Experiment 3 — final refinement
data_tcn3 = prepare_area_data(matrix, PRIMARY_AREA, lookback=288)
tcn_exp3  = run_experiment(
    model_class  = TCNForecaster,
    model_kwargs = {'input_size':1,'num_channels':[64,64,64,64],'kernel_size':5,'dropout':0.3},
    data         = data_tcn3,
    n_epochs=150, lr=5e-4, batch_size=64, patience=15,
    output_dir=EXPERIMENTS_DIR, model_name='TCN_exp03',
)

mt3 = compute_all_metrics(np.array(tcn_exp3['targets_raw']), np.array(tcn_exp3['preds_raw']))
logger.log(
    experiment_id = 'TCN_exp03',
    model_name    = 'TCN',
    config        = {'channels':[64,64,64,64],'kernel':5,'dropout':0.3,'lookback':288,'lr':5e-4},
    metrics       = mt3,
    rationale     = 'Extended lookback to 2 days; reducing LR for finer convergence.',
    next_steps    = 'Final TCN config.',
)

## 4.3 Transformer — Iterative Tuning

In [ ]:
# ============================================================
# Transformer Exp 1 — conservative start
# Pre-LN + low LR to prevent divergence (known instability
# of Transformers with high learning rates on small datasets).
# ============================================================
data_tf1 = prepare_area_data(matrix, PRIMARY_AREA, lookback=144)
tf_exp1  = run_experiment(
    model_class  = TransformerForecaster,
    model_kwargs = {'input_size':1,'d_model':64,'nhead':4,'num_layers':2,'dim_feedfwd':128,'dropout':0.1},
    data         = data_tf1,
    n_epochs=100, lr=1e-4, batch_size=32, patience=10,
    output_dir=EXPERIMENTS_DIR, model_name='Transformer_exp01',
)

mf1 = compute_all_metrics(np.array(tf_exp1['targets_raw']), np.array(tf_exp1['preds_raw']))
logger.log(
    experiment_id = 'TF_exp01',
    model_name    = 'Transformer',
    config        = {'d_model':64,'nhead':4,'layers':2,'ffn':128,'dropout':0.1,'lookback':144,'lr':1e-4},
    metrics       = mf1,
    rationale     = 'Small conservative config with pre-LN. Low LR to avoid early divergence.',
    next_steps    = 'If stable: scale d_model or add layers. If slow to converge: try 5e-4.',
)

In [ ]:
# Transformer Exp 2 — scale up if exp1 was stable
data_tf2 = prepare_area_data(matrix, PRIMARY_AREA, lookback=144)
tf_exp2  = run_experiment(
    model_class  = TransformerForecaster,
    model_kwargs = {'input_size':1,'d_model':128,'nhead':4,'num_layers':3,'dim_feedfwd':256,'dropout':0.1},
    data         = data_tf2,
    n_epochs=100, lr=5e-4, batch_size=32, patience=10,
    output_dir=EXPERIMENTS_DIR, model_name='Transformer_exp02',
)

mf2 = compute_all_metrics(np.array(tf_exp2['targets_raw']), np.array(tf_exp2['preds_raw']))
logger.log(
    experiment_id = 'TF_exp02',
    model_name    = 'Transformer',
    config        = {'d_model':128,'nhead':4,'layers':3,'ffn':256,'dropout':0.1,'lookback':144,'lr':5e-4},
    metrics       = mf2,
    rationale     = 'Scaling d_model 64→128, 2→3 layers. Slightly higher LR given stable exp01.',
    next_steps    = 'Fill after results.',
)

In [ ]:
# Transformer Exp 3 — final
data_tf3 = prepare_area_data(matrix, PRIMARY_AREA, lookback=288)
tf_exp3  = run_experiment(
    model_class  = TransformerForecaster,
    model_kwargs = {'input_size':1,'d_model':128,'nhead':4,'num_layers':3,'dim_feedfwd':256,'dropout':0.2},
    data         = data_tf3,
    n_epochs=150, lr=5e-4, batch_size=32, patience=15,
    output_dir=EXPERIMENTS_DIR, model_name='Transformer_exp03',
)

mf3 = compute_all_metrics(np.array(tf_exp3['targets_raw']), np.array(tf_exp3['preds_raw']))
logger.log(
    experiment_id = 'TF_exp03',
    model_name    = 'Transformer',
    config        = {'d_model':128,'nhead':4,'layers':3,'ffn':256,'dropout':0.2,'lookback':288,'lr':5e-4},
    metrics       = mf3,
    rationale     = 'Extended lookback to 2 days. Slight dropout increase to 0.2 for regularisation.',
    next_steps    = 'Final Transformer config.',
)

# Print full tuning summary
logger.print_summary()

## 4.4 Final Models — Evaluate on All Three Areas

**Before running this cell**: update the `BEST_*` dicts below with the config
that achieved the lowest validation MAE in sections 4.1–4.3 above.

In [ ]:
# ── UPDATE THESE after reviewing tuning results ──────────────────────────────
BEST_LSTM = {
    'model_kwargs': {'input_size':1,'hidden_size':128,'num_layers':2,'dropout':0.3},
    'lr': 5e-4, 'batch_size': 64, 'lookback': 288,
}
BEST_TCN = {
    'model_kwargs': {'input_size':1,'num_channels':[64,64,64,64],'kernel_size':5,'dropout':0.3},
    'lr': 5e-4, 'batch_size': 64, 'lookback': 288,
}
BEST_TF = {
    'model_kwargs': {'input_size':1,'d_model':128,'nhead':4,'num_layers':3,'dim_feedfwd':256,'dropout':0.2},
    'lr': 5e-4, 'batch_size': 32, 'lookback': 288,
}
# ─────────────────────────────────────────────────────────────────────────────

MODEL_CONFIGS = [
    ('LSTM',        LSTMForecaster,        BEST_LSTM),
    ('TCN',         TCNForecaster,         BEST_TCN),
    ('Transformer', TransformerForecaster, BEST_TF),
]

all_results = {}  # {square_id: {model_name: result}}

for sq_id in TARGET_AREAS:
    print(f'\n{"="*60}')
    print(f'Final evaluation  —  Square {sq_id}')
    print(f'{"="*60}')
    area_results = {}

    for model_name, model_class, cfg in MODEL_CONFIGS:
        data = prepare_area_data(matrix, sq_id, lookback=cfg['lookback'])
        result = run_experiment(
            model_class  = model_class,
            model_kwargs = cfg['model_kwargs'],
            data         = data,
            n_epochs     = 150,
            lr           = cfg['lr'],
            batch_size   = cfg['batch_size'],
            patience     = 15,
            output_dir   = EXPERIMENTS_DIR,
            model_name   = f'{model_name}_final_sq{sq_id}',
        )
        area_results[model_name] = result

    all_results[sq_id] = area_results

print('\nAll final experiments complete.')

## 4.5 Results Tables and Plots

Generates:
- **3 results tables** (one per area): MAE, MAPE, RMSE, train time, inference time
- **9 prediction plots** (one per model per area): actual vs predicted Dec 16–22
- **1 failure case plot** per area
- **Training curve plots**

In [ ]:
combined_metrics = run_full_evaluation(all_results, save_dir=FIGURES_DIR)
print('\nCombined Results Table:')
combined_metrics

In [ ]:
# Final summary of timing statistics (Section 4 item IV)
print('Timing Summary')
print('=' * 60)
for sq_id, area_res in all_results.items():
    print(f'\nSquare {sq_id}:')
    for mname, res in area_res.items():
        h = res['history']
        print(f'  {mname:<15} '
              f"train={h['total_train_time_s']:.1f}s  "
              f"({h['epochs_run']} epochs, best={h['best_epoch']})  "
              f"inf={res['inference_time_per_sample_ms']:.3f}ms/sample")